
## 1. Importations et configuration

Nous importons les bibliothèques nécessaires à la manipulation, la visualisation et la modélisation des données :

- **pandas** — chargement, fusion et nettoyage des données

- **numpy** — opérations numériques

- **matplotlib / seaborn** — visualisation des données

- **sklearn** — modèles d'apprentissage automatique

- **warnings** — suppression des avertissements non pertinents

> Vous pouvez installer toute bibliothèque manquante à l'aide de la commande `pip install <nom_de_la_bibliothèque>`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import warnings
from itertools import combinations
from collections import Counter
warnings.filterwarnings('ignore')

### Insight n°4  — À partir de combien de commande un visiteur est considéré comme un client fidèle?

**Question commerciale :**

In [ ]:
df_new = pd.DataFrame(orders)
df_new = df_new[['user_id', 'order_number']]

# Step1 group the data by user
# group the user and count how many rows they have (the numbers of orders)
# Return the max order for each user

group_userID = df_new.groupby('user_id')['order_number'].max()


# Target the sequence column which information should I extract from the group?
# The magic number -- How many unique users reached at least 1 or 2 ... order

# How many user have 1,2 ...... orders.

frequency = (group_userID.value_counts()).sort_index(ascending=False)


#print("The frequency is (max order : number User) ",frequency)


# reach nun in french for nombre utilisateur ayant au moins ce niveau
# Let's calculate the reach the cumulative sum

cumulative_sum = frequency.cumsum()

#let's compare the number of people at Order N+1 to the number of people at Order N.
shift_df = cumulative_sum.shift(1)
prob = shift_df/cumulative_sum
prob_sort20 = (prob.sort_index()).head(19)
#print("The prob sorted is",prob_sort20)


#print("the cumulative sum is ",cumulative_sum)

#print("the percentage (probability) between orders ",prob)


#print("Here's the correction of :",group_userID)

#print("Restricted Orders",restrict_df) reach(n) = 197 358

# Let's visualise




x_axis = prob_sort20.index
y_axis = prob_sort20.values
lineplot = sns.lineplot(x = x_axis, y = y_axis)
plt.axvline(5, color = 'blue', linestyle = "-")
plt.title('Seuil de decision')
plt.xlabel('Nombre de commande')
plt.ylabel('Probabilité de recommander')




### Interprétation

D'après ces données, il existe un « chiffre magique », 5.

La phase de formation de l'habitude : on observe une légère baisse de la fidélisation entre la 1re et la 4e commande. C'est la période où vous êtes le plus susceptible de perdre un client.

Le plateau de fidélité : après la 5e commande, la probabilité de retour cesse de baisser et commence à augmenter régulièrement. Cela indique qu'une fois qu'un client atteint son 5e achat, son habitude de consommation est consolidée et qu'il a atteint un état de fidélité à long terme.

### Stratégie commerciale

Le seuil de décision nous permet de segmenter les clients et d’adapter les actions marketing : inciter les clients proches à acheter, maximiser les clients engagés et optimiser les campagnes de fidélisations et marketing sur les clients peu probables.

### Dans un premier temps :

- Proposer des produits complémentaires
- Faire du cross-selling
- Suggérer des packs
- Récompenser les clients avec des bons de réductions.


---



### Insight n°5  — Quelles sont les produits souvent commandés ensemble?

**Question commerciale :**

In [ ]:
from itertools import combinations
from collections import Counter

df_id = pd.DataFrame(order_products)
df_id  = df_id [['order_id', 'product_id']]
# Étape 1 : Grouper les produits par commande
orders_list = df_id.groupby('order_id')['product_id'].apply(list)

# Étape 2 : Créer les combinaisons de 2 produits par panier
count = Counter()
for products in orders_list:
    # On trie pour éviter d'avoir (A, B) et (B, A) séparément
    combinations_list = combinations(sorted(products), 2)
    count.update(combinations_list) #compter combien de paire qui existe

# Étape 3 : Afficher les 10 duos les plus fréquents
most_common_pairs = count.most_common(10)
for pair, freq in most_common_pairs:
    print(f"Produits: {pair} | Nombre d'associations: {freq}")


plt.figure(figsize=(12,6))
top_products.plot(kind='bar', stacked=True)
plt.xlabel("Top Produits")
plt.ylabel("Nombre d'associations")
plt.title('Produits commander ensemble')
plt.legend(title='Aisle', bbox_to_anchor=(1, 1))
plt.show()


---
## 5. Modèles prédictifs d'apprentissage automatique

La consigne nécessite la création, l'entraînement, l'évaluation et la comparaison d'**au moins deux modèles prédictifs**.

Nous nous concentrons sur la prédiction de la **récommande** d'un produit par un client (`reordered = 1`).
Il s'agit d'un **problème de classification binaire**.

### Caractéristiques utilisées :
- `order_number` — combien de fois l'utilisateur a-t-il déjà passé commande ?
- `order_dow` — quel jour de la semaine la commande a-t-elle été passée ?
- `order_hour_of_day` — à quelle heure la commande a-t-elle été passée ?
- `days_since_prior_order` — combien de jours se sont écoulés depuis la dernière commande ?
- `add_to_cart_order` — à quelle position le produit a-t-il été ajouté au panier ?

### Modèles comparés :
| Modèle | Pourquoi choisi |
|---|---|
| **Régression logistique** | Base simple et interprétable pour la classification binaire |
| **Forêt aléatoire** | Méthode d'ensemble, gère la non-linéarité, fournit l'importance des caractéristiques |


In [ ]:
import pandas as pd

# On identifie le nombre maximum de commandes par utilisateur
user_max_order = orders.groupby('user_id')['order_number'].max().reset_index()

# On crée la cible : 1 si >= 5 commandes, 0 sinon
user_max_order['is_loyal'] = (user_max_order['order_number'] >= 5).astype(int)

# On ne garde que les premières interactions pour l'analyse comportementale
df_start = orders[orders['order_number'] <= 3].copy()

# Feature 1 : Intervalle moyen entre les commandes (Vitesse de retour)
avg_gap = df_start.groupby('user_id')['days_since_prior_order'].mean().reset_index(name='avg_days_gap')

# Feature 2 : Heure de prédilection (Stabilité des habitudes)
std_hour = df_start.groupby('user_id')['order_hour_of_day'].std().reset_index(name='hour_stability')

# Feature 3 : Jour de la semaine préféré
fav_dow = df_start.groupby('user_id')['order_dow'].agg(lambda x: x.mode()[0]).reset_index(name='favorite_day')

# On fusionne tout ça
features = user_max_order[['user_id', 'is_loyal']].merge(avg_gap, on='user_id')
features = features.merge(std_hour, on='user_id').merge(fav_dow, on='user_id')

# On remplace les NaN (ex: std d'une seule commande) par 0 ou 1 a voir
features = features.fillna(0)

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Préparation X et y
X = features.drop(['user_id', 'is_loyal'], axis=1)
y = features['is_loyal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modèle
clf = RandomForestClassifier(n_estimators=100)
clf.fit(X_train, y_train)

# Résultat
print(classification_report(y_test, clf.predict(X_test)))

### Model— Prédiction de la fidélité client

#### Objectif

Prédire si un client deviendra fidèle (au moins 5 commandes) à partir de ses 3 premières commandes.

---

#### Variables utilisées

- **avg_days_gap** : intervalle moyen entre les commandes (engagement)
- **hour_stability** : régularité des heures de commande (habitude)
- **favorite_day** : jour de commande le plus fréquent (routine)

---

#### Méthode

- Création d’une variable cible (fidèle / non fidèle)
- Séparation des données (train / test)
- Entraînement d’un modèle Random Forest
- Évaluation avec un classification report

---

#### Interprétation

Les comportements initiaux permettent d’anticiper la fidélité d’un client.

---

#### Application business

- Identifier rapidement les clients à fort potentiel
- Cibler les clients à risque de churn
- Adapter les actions marketing dès le début